# 06 — Fractal dimension & architectural self-similarity

Reproduces **SI Figure S8** of Eloy et al. *Nat Commun* **8**, 1014
(2017): the quantitative self-similarity figure for one of the evolved
champion genomes.

Five panels:

- **(a)** A mature tree (400 yr) coloured by Strahler order.
- **(b)** Architectural ratios across Horton stream orders: the
  bifurcation ratio $R_n$, the length ratio $R_l$, the diameter ratio
  $R_d$, the area ratio $R_a$, and the fractal dimension
  $D = \log R_n / \log R_l$. Open markers = young tree (25 yr),
  filled markers = mature tree (400 yr).
- **(c)** Time evolution of $R_n$, $R_l$, $D$ from one 500-generation
  simulation.
- **(d)** Branch tapering: per-segment scatter of length vs diameter
  with reference slopes 2/3, 1, 3/2.
- **(e)** Area conservation (Leonardo's rule) at every binary
  bifurcation, mean ≈ 0.95.

Companion of the qualitative diagnostics notebook
[05_strahler_diagnostics.ipynb](05_strahler_diagnostics.ipynb), which
already plots per-Strahler-order counts and the Leonardo histogram. The
new piece here is **per-Horton-stream** length, the four log-linear
fits, and the time evolution of the ratios.


In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mechatree.config import load_config
from mechatree.genome import load_champion
from mechatree.plotting import plot_tree_3d
from mechatree.plotting._palette import strahler_css
from mechatree.simulate import grow_tree
from mechatree.stats import (
    horton_ratios,
    horton_summary,
)

cfg = load_config(Path("../examples/forest.yaml").resolve())
safety, allocation, meta = load_champion(Path("../data/S3_champions.json").resolve(), species_id=0)
SEED = 42
GEN_YOUNG = 25
GEN_MATURE = 400
GEN_MAX = 500
print(f"Champion species_id={meta['species_id']} ({meta.get('centroid_tag', '?')})")

## Setup

We use champion **species 0** from
[`data/S3_champions.json`](../data/S3_champions.json) (swap to
`species_id=1` to compare the second cluster). Two independent
simulations with the same seed:

- **400 generations** → final tree for panel (a) and the *filled*
  markers in panel (b).
- **500 generations with `on_step`** → snapshot history for panel (c)
  and the *open* markers in panel (b) at the 25-yr early snapshot.

The two vertical guides in panel (c) sit at 25 yr and 400 yr — the same
two ages that anchor panel (b)'s open / filled marker pair.


In [ ]:
def snapshot_cb(every: int = 5):
    """Closure: returns an `on_step` callable + a growing history list.

    Every `every` generations we snapshot the Horton stream summary;
    panel (c) replays the history to plot R_n(t), R_l(t), D(t)."""
    history: list[dict] = []

    def cb(gen: int, tree) -> None:
        if gen % every == 0:
            history.append({"gen": gen, "summary": horton_summary(tree)})

    return cb, history


def per_branch_arrays(tree):
    """Per-segment (strahler, length, diameter) arrays for panel (d)."""
    tree.set_strahler()
    n = tree.get_number_of_branches()
    strahler = np.empty(n, dtype=np.int32)
    length = np.empty(n, dtype=np.float64)
    diameter = np.empty(n, dtype=np.float64)
    for i in range(n):
        strahler[i] = tree.get_strahler(i)
        length[i] = tree.get_length(i)
        diameter[i] = tree.get_diameter(i)
    return strahler, length, diameter


def bifurcation_table(tree):
    """Panel (e): per-binary-junction (parent_length, area_ratio).

    `area_ratio = (d_left^2 + d_right^2) / d_parent^2` — the
    cross-section ratio that should be ≈ 1 if Leonardo's rule holds."""
    n = tree.get_number_of_branches()
    rows = []
    for i in range(n):
        kids = tree.get_children_index(i)
        if len(kids) != 2:
            continue
        d_p = tree.get_diameter(i)
        if d_p <= 0.0:
            continue
        d_a = tree.get_diameter(kids[0])
        d_b = tree.get_diameter(kids[1])
        rows.append((tree.get_length(i), (d_a * d_a + d_b * d_b) / (d_p * d_p)))
    return np.array(rows, dtype=[("length", "f8"), ("ratio", "f8")])

## Panel (a) — Mature tree at 400 yr

Grow the champion once for 400 generations and render the canopy
coloured by Strahler order via the existing `plot_tree_3d` helper.


In [ ]:
tree_mature = grow_tree(
    cfg,
    n_generations=GEN_MATURE,
    seed=SEED,
    safety=safety,
    allocation=allocation,
)
print(
    f"{GEN_MATURE} yr tree: {tree_mature.get_number_of_branches()} branches, "
    f"max Strahler = {horton_summary(tree_mature).max_order}"
)
# width_max bumped above the 6-px default so the trunk reads visibly
# fat. plot_tree_3d still uses pixel-width lines (not Mesh3d cylinders),
# so this is an approximation of the Blender renders in the paper.
fig_a = plot_tree_3d(tree_mature, width_max=18.0)
fig_a.update_layout(title=f"{GEN_MATURE} yr champion (species_id={meta['species_id']})")
fig_a.show()

## Panels (b)–(e) — A second run with per-generation snapshots

We re-grow with the same seed but for 500 generations, snapshotting
`horton_summary` every 5 generations. The 25-yr snapshot is the open
markers in panel (b); the 400-yr `tree_mature` from above contributes
the filled markers, the tapering scatter in panel (d) and the
bifurcation table in panel (e).


In [ ]:
on_step, history = snapshot_cb(every=5)
tree_500 = grow_tree(
    cfg,
    n_generations=GEN_MAX,
    seed=SEED,
    safety=safety,
    allocation=allocation,
    on_step=on_step,
)

summary_400 = horton_summary(tree_mature)
# Pick the snapshot whose generation is closest to GEN_YOUNG.
summary_25 = min(history, key=lambda h: abs(h["gen"] - GEN_YOUNG))["summary"]

fit_400 = horton_ratios(summary_400)
fit_25 = horton_ratios(summary_25)
print("Paper targets:  R_n=3.5  R_l=1.7  R_d=1.9  R_a=3.5  D=2.3")
print(
    f" 25 yr fit:    R_n={fit_25.R_n:.2f}  R_l={fit_25.R_l:.2f}  "
    f"R_d={fit_25.R_d:.2f}  R_a={fit_25.R_a:.2f}  D={fit_25.D:.2f}"
)
print(
    f"400 yr fit:    R_n={fit_400.R_n:.2f}  R_l={fit_400.R_l:.2f}  "
    f"R_d={fit_400.R_d:.2f}  R_a={fit_400.R_a:.2f}  D={fit_400.D:.2f}"
)

### Build the 2×2 panel

Plotly does not mix 3D and 2D subplots cleanly, so panel (a) lives in
its own figure above. Panels (b)–(e) share one `make_subplots` grid.


In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "(b) Architectural self-similarity",
        "(c) Evolution of fractal dimension",
        "(d) Branch tapering",
        "(e) Area conservation (Leonardo)",
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.14,
)


# ------------- Panel (b): per-order quantities + fits -------------------
def _series_b(summary, name_suffix, symbol_open=False, showlegend=True):
    w = np.arange(1, summary.max_order + 1)
    marker_style = dict(size=9, line=dict(width=1.2))
    if symbol_open:
        # Open markers: only the outline, white fill.
        marker_style["color"] = "white"
    trace_specs = [
        ("# branches", summary.n_branches, "circle", "crimson"),
        ("mean length", summary.mean_length, "diamond", "royalblue"),
        ("mean area", summary.mean_area, "triangle-up", "indianred"),
        ("mean diameter", summary.mean_diameter, "triangle-down", "lightsteelblue"),
    ]
    for label, y, sym, line_color in trace_specs:
        marker = dict(marker_style)
        marker["symbol"] = sym
        marker["line"] = dict(width=1.4, color=line_color)
        if not symbol_open:
            marker["color"] = line_color
        fig.add_trace(
            go.Scatter(
                x=w,
                y=y,
                mode="markers",
                name=f"{label} {name_suffix}",
                marker=marker,
                showlegend=showlegend,
                legendgroup=label,
            ),
            row=1,
            col=1,
        )


_series_b(summary_25, "(25 yr)", symbol_open=True, showlegend=True)
_series_b(summary_400, "(400 yr)", symbol_open=False, showlegend=True)

# Reference fit lines from the mature tree, drawn over fit_orders.
w_fit = fit_400.fit_orders
fit_lines = [
    (summary_400.n_branches, fit_400.R_n, -1, "crimson"),
    (summary_400.mean_length, fit_400.R_l, +1, "royalblue"),
    (summary_400.mean_area, fit_400.R_a, +1, "indianred"),
    (summary_400.mean_diameter, fit_400.R_d, +1, "lightsteelblue"),
]
for arr, ratio, sign, color in fit_lines:
    # Anchor the line at w=1 using the actual data point, then propagate
    # geometrically by `ratio` per rank step in the sign direction.
    anchor = float(arr[0])
    yline = anchor * ratio ** (sign * (w_fit - 1))
    fig.add_trace(
        go.Scatter(
            x=w_fit,
            y=yline,
            mode="lines",
            line=dict(color=color, width=1, dash="dash"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

fig.add_annotation(
    xref="x",
    yref="y",
    x=1.2,
    y=summary_400.n_branches[0] * 0.35,
    text=(
        f"R_n = {fit_400.R_n:.2f}<br>"
        f"R_l = {fit_400.R_l:.2f}<br>"
        f"R_d = {fit_400.R_d:.2f}<br>"
        f"R_a = {fit_400.R_a:.2f}<br>"
        f"D   = {fit_400.D:.2f}"
    ),
    showarrow=False,
    align="left",
    font=dict(size=10),
    bgcolor="rgba(255,255,255,0.7)",
    row=1,
    col=1,
)
fig.update_xaxes(title_text="Horton order w", row=1, col=1)
fig.update_yaxes(title_text="value", type="log", row=1, col=1)

# ------------- Panel (c): time evolution of R_n, R_l, D -----------------
t_arr, R_n_arr, R_l_arr, D_arr = [], [], [], []
R_n_band, R_l_band, D_band = [], [], []
for h in history:
    s = h["summary"]
    if s.max_order < 4:
        continue
    try:
        fit = horton_ratios(s)
    except (ValueError, ZeroDivisionError):
        continue
    t_arr.append(h["gen"])
    R_n_arr.append(fit.R_n)
    R_l_arr.append(fit.R_l)
    D_arr.append(fit.D)
    # Per-rank pair scatter as a uncertainty band.
    last = s.max_order - 1
    n_pairs = s.n_branches[: last - 1] / np.where(s.n_branches[1:last] > 0, s.n_branches[1:last], 1)
    l_pairs = s.mean_length[1:last] / np.where(
        s.mean_length[: last - 1] > 0, s.mean_length[: last - 1], 1
    )
    R_n_band.append(np.std(n_pairs[np.isfinite(n_pairs) & (n_pairs > 0)]))
    R_l_band.append(np.std(l_pairs[np.isfinite(l_pairs) & (l_pairs > 0)]))
    # D's uncertainty propagates from R_n and R_l (rough).
    D_band.append(
        abs(fit.D)
        * np.sqrt(
            (R_n_band[-1] / max(fit.R_n, 1e-9) / np.log(max(fit.R_n, 1.001))) ** 2
            + (R_l_band[-1] / max(fit.R_l, 1e-9) / np.log(max(fit.R_l, 1.001))) ** 2
        )
    )

t_arr = np.asarray(t_arr)
for label, y, band, color in [
    ("R_n", R_n_arr, R_n_band, "crimson"),
    ("R_l", R_l_arr, R_l_band, "royalblue"),
    ("D", D_arr, D_band, "black"),
]:
    y = np.asarray(y)
    band = np.asarray(band)
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([t_arr, t_arr[::-1]]),
            y=np.concatenate([y + band, (y - band)[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=0.12,
            line=dict(color="rgba(0,0,0,0)"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=t_arr,
            y=y,
            mode="lines+markers",
            name=label,
            line=dict(color=color, width=2),
            marker=dict(size=4),
        ),
        row=1,
        col=2,
    )

for gen, txt in [(GEN_YOUNG, "25 yr"), (GEN_MATURE, "400 yr")]:
    fig.add_vline(
        x=gen,
        line=dict(color="gray", width=1, dash="dot"),
        annotation_text=txt,
        annotation_position="top",
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="time (yr)", row=1, col=2)
fig.update_yaxes(title_text="ratio / dimension", row=1, col=2)

# ------------- Panel (d): branch tapering -------------------------------
strahler, length, diameter = per_branch_arrays(tree_mature)
mask = (length > 0) & (diameter > 0)
strahler = strahler[mask]
length = length[mask]
diameter = diameter[mask]

for k in range(1, int(strahler.max()) + 1):
    sub = strahler == k
    if not sub.any():
        continue
    fig.add_trace(
        go.Scatter(
            x=diameter[sub],
            y=length[sub],
            mode="markers",
            marker=dict(size=4, color=strahler_css(k), opacity=0.6, line=dict(width=0)),
            name=f"order {k}",
            legendgroup=f"d-order-{k}",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

# Reference slopes through the cloud centroid.
log_d_mid = np.log10(np.median(diameter))
log_l_mid = np.log10(np.median(length))
d_ref = np.array([diameter.min(), diameter.max()])
for slope, label in [(2 / 3, "2/3"), (1.0, "1"), (3 / 2, "3/2")]:
    l_ref = 10 ** (log_l_mid + slope * (np.log10(d_ref) - log_d_mid))
    fig.add_trace(
        go.Scatter(
            x=d_ref,
            y=l_ref,
            mode="lines",
            line=dict(color="black", width=1, dash="dash"),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=2,
        col=1,
    )
    fig.add_annotation(
        x=d_ref[-1],
        y=l_ref[-1],
        text=f"slope {label}",
        showarrow=False,
        font=dict(size=9),
        xanchor="left",
        row=2,
        col=1,
    )
fig.update_xaxes(title_text="diameter d", type="log", row=2, col=1)
fig.update_yaxes(title_text="length ⟨ℓ⟩", type="log", row=2, col=1)

# ------------- Panel (e): area conservation -----------------------------
tbl = bifurcation_table(tree_mature)
mean_ratio = float(tbl["ratio"].mean()) if tbl.size else float("nan")
fig.add_trace(
    go.Scatter(
        x=tbl["length"],
        y=tbl["ratio"],
        mode="markers",
        marker=dict(size=3, color="royalblue", opacity=0.35, line=dict(width=0)),
        showlegend=False,
        name="A_kids / A_parent",
    ),
    row=2,
    col=2,
)
fig.add_hline(
    y=mean_ratio,
    line=dict(color="black", width=1.5),
    annotation_text=f"mean = {mean_ratio:.2f}",
    annotation_position="bottom right",
    row=2,
    col=2,
)
fig.add_hline(
    y=1.0,
    line=dict(color="gray", width=1, dash="dot"),
    row=2,
    col=2,
)
fig.update_xaxes(title_text="parent length ⟨ℓ⟩", row=2, col=2)
fig.update_yaxes(title_text="area ratio  (A_left + A_right) / A_parent", row=2, col=2)

fig.update_layout(
    height=820,
    width=1000,
    title=f"SI Fig. S8 — champion species_id={meta['species_id']}, seed={SEED}",
    legend=dict(font=dict(size=10)),
    template="plotly_white",
)
fig.show()

## Discussion

The 400-yr fitted ratios should sit within roughly ±20 % of the paper
targets `R_n = 3.5`, `R_l = 1.7`, `R_d = 1.9`, `R_a = 3.5`, `D = 2.3`
— close enough to confirm that the Python port recovers the published
self-similarity. Single-seed scatter is real; averaging across seeds
tightens the bands.

Panel (b) shows the qualitative point of the paper directly: the
*open* markers (25 yr, juvenile) lie off the geometric trend lines
while the *filled* markers (400 yr, mature) line up. Panel (c) traces
that same convergence over time — the ratios drift toward their
asymptotic values within ~200 generations.

Panel (e) typically lands near 0.95: cross-section is not perfectly
conserved at branchings in this model, but it is close. The horizontal
gray dotted line at 1.0 marks the strict Leonardo limit; the deficit
reflects the model's preference for slightly thinner daughters.


## Next steps

- Swap to `species_id=1` in cell 1 and re-run to compare the two S3
  champion clusters side-by-side.
- Re-run with the canopy-aware DendroFlow wind by adding a
  ``wind: { model: dendroflow, ... }`` block to a forked
  `forest.yaml` — `grow_tree` picks it up automatically.
- Sweep `SEED` over 10–20 values and aggregate `fit_400` to get true
  population bands on each ratio.
- Compare against `notebooks/05_strahler_diagnostics.ipynb` for the
  per-Strahler-order (per-segment) view, which is appropriate for the
  Leonardo histogram but cannot resolve length ratios since every
  segment in MechaTree is a unit twig.
